# Heteroskedasticity and Robust Standard Errors

**DS4DH · Module 05 — Regression Analysis**

*Technique:* Detecting non-constant residual variance and refitting with HC1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/05c_robust_se.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

OLS coefficients are unbiased even when the residual variance is not constant.
The **standard errors** are not. That means the coefficient is fine and every
p-value and confidence interval built from it is wrong.

Housing data is heteroskedastic almost by construction: large, diverse rental
markets vary more than small uniform ones.

In [ ]:
csd = df.dropna(subset=['csd_code'])

reg_df = csd[csd['immigrant_status'].isin(['Immigrant', 'Non-immigrants'])
             & csd['cma'].isin(CITIES)].dropna(subset=['Renter']).copy()
reg_df['is_immigrant'] = (reg_df['immigrant_status'] == 'Immigrant').astype(int)

print(f'{len(reg_df)} rows — one per (CSD, immigrant status) with a renter STIR')
print(reg_df['immigrant_status'].value_counts().to_string())

In [ ]:
city_dummies = pd.get_dummies(reg_df['cma'], drop_first=True, dtype=float)
y = reg_df['Renter']
X = sm.add_constant(pd.concat([reg_df[['is_immigrant']], city_dummies], axis=1))

model = sm.OLS(y, X).fit()
reg_df = reg_df.assign(resid=model.resid, fitted=model.fittedvalues)

print(f'{"City":<12}{"n":>5}{"residual sd":>14}')
print('-' * 31)
for city in CITIES:
    r = reg_df[reg_df['cma'] == city]['resid']
    print(f'{city:<12}{len(r):>5}{r.std():>14.2f}')
print()
print('If these were equal, the constant-variance assumption would hold.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(reg_df['fitted'], reg_df['resid'], alpha=0.5, s=18)
axes[0].axhline(0, color='#E8663D', lw=1.5)
axes[0].set_xlabel('fitted value')
axes[0].set_ylabel('residual')
axes[0].set_title('Residuals vs fitted — look for a fan shape')

groups = [reg_df[reg_df['cma'] == c]['resid'] for c in CITIES]
axes[1].boxplot(groups)
axes[1].set_xticklabels(CITIES)
axes[1].axhline(0, color='#E8663D', lw=1.5)
axes[1].set_ylabel('residual')
axes[1].set_title('Residual spread differs by city')

plt.tight_layout()
plt.show()

## Testing it formally

The Breusch–Pagan test regresses squared residuals on the predictors. A small
p-value says the variance depends on them — heteroskedasticity is present.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

lm, lm_p, f, f_p = het_breuschpagan(model.resid, model.model.exog)
print(f'Breusch-Pagan LM statistic : {lm:.3f}')
print(f'                    p-value : {lm_p:.5f}')
print()
print('heteroskedastic' if lm_p < 0.05 else 'no strong evidence of heteroskedasticity')

## The fix

Refit with `cov_type='HC1'`. The coefficients do not move — they were never the
problem. Only the standard errors, and everything derived from them, change.

In [ ]:
robust = sm.OLS(y, X).fit(cov_type='HC1')

print(f'{"":<16}{"default":>12}{"HC1 robust":>13}{"change":>10}')
print('-' * 51)
print(f'{"coefficient":<16}{model.params["is_immigrant"]:>12.4f}'
      f'{robust.params["is_immigrant"]:>13.4f}'
      f'{robust.params["is_immigrant"] - model.params["is_immigrant"]:>10.4f}')
print(f'{"std error":<16}{model.bse["is_immigrant"]:>12.4f}'
      f'{robust.bse["is_immigrant"]:>13.4f}'
      f'{robust.bse["is_immigrant"] - model.bse["is_immigrant"]:>+10.4f}')
print(f'{"p-value":<16}{model.pvalues["is_immigrant"]:>12.4f}'
      f'{robust.pvalues["is_immigrant"]:>13.4f}'
      f'{robust.pvalues["is_immigrant"] - model.pvalues["is_immigrant"]:>+10.4f}')
print()
print('Coefficients identical to four decimals. Standard error slightly larger.')

In [ ]:
# The confidence interval is what a reader should see.
ci_d = model.conf_int().loc['is_immigrant']
ci_r = robust.conf_int().loc['is_immigrant']
print(f'default 95% CI : [{ci_d[0]:+.3f}, {ci_d[1]:+.3f}] pp')
print(f'HC1     95% CI : [{ci_r[0]:+.3f}, {ci_r[1]:+.3f}] pp')
print()
print('Both include zero. The robust interval is the honest one, and it is')
print('wide enough to contain effects in both directions.')

### 🔧 Your turn 1

Try `cov_type='HC3'` instead of `HC1`.

HC3 is more conservative and generally preferred for smaller samples. How much
wider does the interval get? Would either choice change your conclusion?

### 🔧 Your turn 2

Cluster the standard errors by city instead:

```python
sm.OLS(y, X).fit(cov_type='cluster',
                 cov_kwds={'groups': reg_df['cma']})
```

With only four clusters this is a bad idea — but run it and see what happens to
the standard error. Why does a small number of clusters make clustering unsafe?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** HC3 gives a slightly wider interval again. Neither changes the
conclusion, because the coefficient was nowhere near significant to begin with —
the interval comfortably spans zero under every variance estimator. When a
conclusion is stable across HC1, HC3 and the default, say so; it is a genuine
robustness result and costs one sentence.

**Your turn 2.** The clustered standard error is much larger and unreliable.
Cluster-robust inference is asymptotic in the *number of clusters*, not the number
of observations — the usual rule of thumb is at least 30–50. With four cities the
estimator has almost no information about between-cluster variation, and it will
be badly biased in a direction you cannot predict. Fixed effects for four cities
are fine; clustering on four cities is not.

</details>

## Where this stops

Module 05's conclusion: after controlling for city, the immigrant renter gap is
about +0.27pp and is not distinguishable from zero under any reasonable variance
estimator. The only robust result in this dataset remains Edmonton's negative gap
from Module 04.

Next: Module 06 stops asking about this one comparison and looks for structure in
the data that nobody labelled.